# Block 7 — Hyperparameter Tuning with Optuna

**Goals for this block:**

- Learn automated hyperparameter optimization with **Optuna**
- Tune a **Temporal Convolutional Network (TCN)** model
- Find optimal parameters for **TCN** with **future covariates**: calendar features (hour, day-of-week, month)
- Compare performance improvements through systematic tuning

## 0. Environment & Imports

In [ ]:
import tensorflow as tf
gpus = tf.config.list_physical_devices('GPU')
tf.config.set_logical_device_configuration(gpus[0], [tf.config.LogicalDeviceConfiguration(memory_limit=6500)])

import torch
assert torch.cuda.is_available()
torch.cuda.set_per_process_memory_fraction(0.3)

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.preprocessing import MinMaxScaler

from darts.dataprocessing.transformers import Scaler
from darts.utils.missing_values import fill_missing_values


## 2. Load & Prepare Data

Load the electricity consumption dataset and prepare it for time series forecasting.

In [ ]:
from darts.datasets import ElectricityConsumptionZurichDataset

series = ElectricityConsumptionZurichDataset().load() 
target_columns = ["Value_NE5", "Value_NE7"]
covariate_columns = ['Hr [%Hr]', 'RainDur [min]', 'StrGlo [W/m2]', 'T [°C]', 'WD [°]', 'WVs [m/s]', 'WVv [m/s]', 'p [hPa]']

# splt the time series into target and covariates
target_series = series[target_columns]
past_covariates = series[covariate_columns]


<div style="background-color: #ffedcc; border-left: 6px solid #ff7518; padding: 10px; margin-bottom: 10px;">
<h2>📝 Exercise: Prepare data for training</h2>

1. Preprocess the dataset using our preprocessing pipeline
    - create now a train, **validation** and test sets


</div>

#### 2.1 Basic Preprocessing

Apply our standard preprocessing pipeline to the electricity data.

⚠️: We will be tuning a TCN model which takes past covariates. Think of preprocessing target and past covariate series

In [ ]:
# splt the time series into 70% train, 15% validation, 15% test


In [ ]:
# preprocess target series


# preprocess covariate series


<div style="background-color: #ffedcc; border-left: 6px solid #ff7518; padding: 10px; margin-bottom: 10px;">
<h2>📝 Exercise: Hyperparameter Optimization with Optuna</h2>

Learn how to systematically find the best model configuration using Optuna's optimization framework.

1. Build the objective function that defines what to optimize

2. Run a tuning job to find optimal hyperparameters

This exercise will teach you automated hyperparameter optimization for time series models.

</div>

### 1. **Study**: The optimization session
```python
study = optuna.create_study(direction="minimize", study_name="LSTM_sMAPE")
```
- `direction`: "minimize" (for loss/error) or "maximize" (for accuracy)
- `study_name`: Identifier for tracking experiments

#### 2. **Trial**: Single hyperparameter configuration attempt
```python
def objective_lstm(trial):
    # Suggest hyperparameter values
    window_size = 24 * trial.suggest_int("window_size_factor", 1, 7)
    hidden_dim = trial.suggest_categorical("hidden_dim", [32, 64, 96, 128])
    
    # Train model with suggested parameters
    model = RNNModel(input_chunk_length=window_size, hidden_dim=hidden_dim, ...)
    
    # Return metric to optimize
    return validation_error
```

#### 3. **Hyperparameter Suggestion Methods**

| Method | Use Case | Example |
|--------|----------|---------|
| `suggest_int(name, low, high)` | Integer ranges | `trial.suggest_int("layers", 1, 5)` |
| `suggest_float(name, low, high)` | Continuous values | `trial.suggest_float("lr", 1e-5, 1e-1, log=True)` |
| `suggest_categorical(name, choices)` | Discrete choices | `trial.suggest_categorical("optimizer", ["adam", "sgd"])` |
| `suggest_uniform(name, low, high)` | Uniform distribution | `trial.suggest_uniform("dropout", 0.1, 0.5)` |


### 3.2 TCN Objective Function
Define the objective function for optimizing our Temporal Convolutional Network.

In [ ]:
# Fixed forecast horizon for all experiments
forecast_horizon = 24

Suggestion: TCN Hyperparameters to be tuned:

- `input_chunk_length`: How many past timesteps to use (4-10 days)
- `num_filters`: Number of convolutional filters (2, 4, 16)
- `kernel_size`: Size of convolutional kernels (3 to 11)
- `dilation_base`: Base for exponential dilation (2 to 4)

In [ ]:
from darts.models import TCNModel
from darts.metrics import smape

# 1. Build the objective function
def objective_tcn(trial):
    model = TCNModel(
                ...
            )

    # Train the model without using validation covariates
    model.fit(
                ...
            )
    
    # Generate historical forecasts on validation set
    pred_series = model.historical_forecasts(...)
    
    # Inverse transform predictions and targets
    pred_unscaled = ...

    # Calculate and return sMAPE
    smape_score = ...
    return smape_score

## 4. Running Optimization Studies

Execute the hyperparameter optimization and analyze results.

### 4.1 Optimize TCN Parameters

In [ ]:
import optuna

study_tcn = ...

...  # Create Optuna study for TCN model

print("Best TCN sMAPE:", round(study_tcn.best_value, 4))
print(study_tcn.best_trial.params)


### 4.2 Analyze Optimization Results
Review all trials to understand hyperparameter impact on performance. 

`value`refers to the metric you choose to optimize (in this case sMAPE)

In [ ]:
df_results = study_tcn.trials_dataframe().sort_values("value")
df_results.head(15)

This table shows:

- value: Validation metric sMAPE (lower is better)

- params_*: Hyperparameter values for each trial

- Results sorted by performance (best first)

## 5. Final Evaluation on Test

Let's evaluate our best model and compare its performance with the TCN model from BLOCK 3

### 5.1 Train Best Model on Full Training Data


In [ ]:
from darts.metrics import mae

# Rebuild the best model from the best trial
best_tcn = TCNModel(
            ...
        )

# Combine train and validation for final training
trainval_scaled = train_scaled.append(val_scaled)
trainval_past_cov = train_cov_scaled.append(val_cov_scaled)

# Train final models on train+val data
best_tcn.fit(...)


# Make predictions on test set
pred_tcn = best_tcn.historical_forecasts(...)

# Inverse transform predictions and test data
pred_tcn_unscaled =  ...

### 5.2 Performance Analysis & Visualization


In [ ]:
# Calculate MAE, RMSE, MAPE, sMAPE

from darts.metrics import mae, rmse, mape, smape

evaluation_metrics_tcn = {
    "MAE": mae(test_target, pred_tcn_unscaled),
    "RMSE": rmse(test_target, pred_tcn_unscaled),
    "MAPE": mape(test_target, pred_tcn_unscaled),
    "sMAPE": smape(test_target, pred_tcn_unscaled),
}
print(evaluation_metrics_tcn)

In [ ]:
# Plotting some predictions vs actuals
start_idx = pd.Timestamp('2021-07-25 03:00:00')
end_idx = pd.Timestamp('2021-08-02 03:00:00')

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(12, 10))
test_target[target_columns[0]][start_idx:end_idx].plot(label="Actual", color='blue', ax=ax1)
pred_tcn_unscaled[target_columns[0]][start_idx:end_idx].plot(label="TCN", color='red', title=f"Actual vs Predicted Energy Consumption {target_columns[0]}", ax=ax1)

test_target[target_columns[1]][start_idx:end_idx].plot(label="Actual", color='blue', ax=ax2)
pred_tcn_unscaled[target_columns[1]][start_idx:end_idx].plot(label="TCN", color='red', title=f"Actual vs Predicted Energy Consumption {target_columns[1]}", ax=ax2)

ax1.legend()
ax2.legend()
plt.tight_layout()
plt.show()

## ✅ Summary

## What You've Accomplished:

- Automated Hyperparameter Optimization: Used Optuna to systematically find optimal model configurations

- TCN Model Tuning: Optimized Temporal Convolutional Network parameters for time series forecasting

- Performance Comparison: Evaluated improvements from systematic tuning vs. manual parameter selection
